In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
import xgboost as xgb
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Load your dataset
file_path = "/content/t5_embeddings_abstractive (1).xlsx" # Replace with the path to your dataset
dataset = pd.read_excel(file_path)

# Convert column names to strings to avoid errors
dataset.columns = dataset.columns.astype(str)

# Separate features (X) and target (y)
X = dataset.drop(columns=['Judgement Status'])
y = dataset['Judgement Status']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the data (necessary for KNN and SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define classifiers
classifiers = {
    'XGBoost': xgb.XGBClassifier(eval_metric='mlogloss'),
    'AdaBoost': AdaBoostClassifier(algorithm='SAMME'),
    'RandomForest': RandomForestClassifier(),
    'DecisionTree': DecisionTreeClassifier(),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC()
}

# Train and evaluate each model
metrics = {}

for model_name, model in classifiers.items():
    # Fit model on training data
    if model_name in ['KNN', 'SVM']:
        model.fit(X_train_scaled, y_train)
        y_train_pred = model.predict(X_train_scaled)
        y_test_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

    # Calculate train and test metrics
    metrics[model_name] = {
        'Train Accuracy': accuracy_score(y_train, y_train_pred),
        'Test Accuracy': accuracy_score(y_test, y_test_pred),
        'Train F1-Score': f1_score(y_train, y_train_pred, average='weighted'),
        'Test F1-Score': f1_score(y_test, y_test_pred, average='weighted')
    }

# Convert metrics to a DataFrame
metrics_df = pd.DataFrame(metrics).T

# Save the metrics to an Excel file
output_path = "model_evaluation_metrics_abswithoutpca.xlsx"  # Define the path where the Excel file will be saved
metrics_df.to_excel(output_path, index=True)

print("Model evaluation metrics have been saved to:", output_path)


Model evaluation metrics have been saved to: model_evaluation_metrics_abswithoutpca.xlsx


In [6]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.decomposition import PCA
import xgboost as xgb
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
import numpy as np

# Load your dataset
file_path =  "/content/t5_embeddings_abstractive (1).xlsx"  # Replace with the path to your dataset
dataset = pd.read_excel(file_path)

# Convert column names to strings to avoid errors
dataset.columns = dataset.columns.astype(str)

# Separate features (X) and target (y)
X = dataset.drop(columns=['Judgement Status'])
y = dataset['Judgement Status']

# Apply SMOTE for class balancing
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply PCA (reduce dimensionality to 20 components)
pca = PCA(n_components=20)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Define classifiers with parameters to reduce overfitting
classifiers = {
    'XGBoost': xgb.XGBClassifier(n_estimators=200, learning_rate=0.01, max_depth=3, eval_metric='mlogloss', n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(n_estimators=200, learning_rate=0.01),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=5, min_samples_leaf=2, n_jobs=-1),
    'DecisionTree': DecisionTreeClassifier(max_depth=10, min_samples_split=5, min_samples_leaf=2),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'SVM': SVC(kernel='rbf', C=1, gamma=0.01)
}

# Use Stratified K-Folds Cross-Validation
skf = StratifiedKFold(n_splits=5)
metrics = {}

for model_name, model in classifiers.items():
    model.fit(X_train_pca, y_train)

    # Predict on training and test data
    y_train_pred = model.predict(X_train_pca)
    y_test_pred = model.predict(X_test_pca)

    # Calculate metrics
    metrics[model_name] = {
        'Train Accuracy': accuracy_score(y_train, y_train_pred),
        'Test Accuracy': accuracy_score(y_test, y_test_pred),
        'Train Recall': recall_score(y_train, y_train_pred, average='weighted'),
        'Test Recall': recall_score(y_test, y_test_pred, average='weighted'),
        'Train F1-Score': f1_score(y_train, y_train_pred, average='weighted'),
        'Test F1-Score': f1_score(y_test, y_test_pred, average='weighted')
    }

# Display the metrics for each model
metrics_df = pd.DataFrame(metrics).T
# Save the metrics to an Excel file
output_path = "model_evaluation_metrics_abswithpca.xlsx"  # Define the path where the Excel file will be saved
metrics_df.to_excel(output_path, index=True)

print("Model evaluation metrics have been saved to:", output_path)


/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Model evaluation metrics have been saved to: model_evaluation_metrics_abswithpca.xlsx


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import xgboost as xgb
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Load your dataset
file_path ="/content/t5_embeddings_extractive (1).xlsx"  # Replace with the path to your dataset
dataset = pd.read_excel(file_path)

# Convert column names to strings to avoid errors
dataset.columns = dataset.columns.astype(str)

# Separate features (X) and target (y)
X = dataset.drop(columns=['Judgement Status'])
y = dataset['Judgement Status']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the data (necessary for KNN and SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define classifiers (with fixes for warnings)
classifiers = {
    'XGBoost': xgb.XGBClassifier(eval_metric='mlogloss'),  # Removed use_label_encoder
    'AdaBoost': AdaBoostClassifier(algorithm='SAMME'),  # Specified SAMME to avoid warning
    'RandomForest': RandomForestClassifier(),
    'DecisionTree': DecisionTreeClassifier(),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC()
}

# Train and evaluate each model
metrics = {}

for model_name, model in classifiers.items():
    # Fit model on training data
    if model_name in ['KNN', 'SVM']:
        model.fit(X_train_scaled, y_train)
        y_train_pred = model.predict(X_train_scaled)
        y_test_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

    # Calculate train and test metrics
    metrics[model_name] = {
        'Train Accuracy': accuracy_score(y_train, y_train_pred),
        'Test Accuracy': accuracy_score(y_test, y_test_pred),
        'Train F1-Score': f1_score(y_train, y_train_pred, average='weighted'),
        'Test F1-Score': f1_score(y_test, y_test_pred, average='weighted')
    }

# Display the metrics for each model
metrics_df = pd.DataFrame(metrics).T
# Save the metrics to an Excel file
output_path = "model_evaluation_metrics_extwithoutpca.xlsx"  # Define the path where the Excel file will be saved
metrics_df.to_excel(output_path, index=True)

print("Model evaluation metrics have been saved to:", output_path)


Model evaluation metrics have been saved to: model_evaluation_metrics_extwithoutpca.xlsx


In [8]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.decomposition import PCA
import xgboost as xgb
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
import numpy as np

# Load your dataset
file_path = "/content/t5_embeddings_extractive (1).xlsx"  # Replace with the path to your dataset
dataset = pd.read_excel(file_path)

# Convert column names to strings to avoid errors
dataset.columns = dataset.columns.astype(str)

# Separate features (X) and target (y)
X = dataset.drop(columns=['Judgement Status'])
y = dataset['Judgement Status']

# Apply SMOTE for class balancing
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply PCA (reduce dimensionality to 20 components)
pca = PCA(n_components=20)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Define classifiers with parameters to reduce overfitting
classifiers = {
    'XGBoost': xgb.XGBClassifier(n_estimators=200, learning_rate=0.01, max_depth=3, eval_metric='mlogloss', n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(n_estimators=200, learning_rate=0.01),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=5, min_samples_leaf=2, n_jobs=-1),
    'DecisionTree': DecisionTreeClassifier(max_depth=10, min_samples_split=5, min_samples_leaf=2),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'SVM': SVC(kernel='rbf', C=1, gamma=0.01)
}

# Use Stratified K-Folds Cross-Validation
skf = StratifiedKFold(n_splits=5)
metrics = {}

for model_name, model in classifiers.items():
    model.fit(X_train_pca, y_train)

    # Predict on training and test data
    y_train_pred = model.predict(X_train_pca)
    y_test_pred = model.predict(X_test_pca)

    # Calculate metrics
    metrics[model_name] = {
        'Train Accuracy': accuracy_score(y_train, y_train_pred),
        'Test Accuracy': accuracy_score(y_test, y_test_pred),
        'Train Recall': recall_score(y_train, y_train_pred, average='weighted'),
        'Test Recall': recall_score(y_test, y_test_pred, average='weighted'),
        'Train F1-Score': f1_score(y_train, y_train_pred, average='weighted'),
        'Test F1-Score': f1_score(y_test, y_test_pred, average='weighted')
    }

# Display the metrics for each model
metrics_df = pd.DataFrame(metrics).T
# Save the metrics to an Excel file
output_path = "model_evaluation_metrics_extwithpca.xlsx"  # Define the path where the Excel file will be saved
metrics_df.to_excel(output_path, index=True)

print("Model evaluation metrics have been saved to:", output_path)


/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Model evaluation metrics have been saved to: model_evaluation_metrics_extwithpca.xlsx
